### Завдання 1. Підбір схожих фільмів для нової стрічки
- Ситуація: Кіностудія випустила новий фільм, і потрібно запропонувати глядачам інші фільми, які їм можуть сподобатися.
- Кроки:
1. Завантажити датасет The Movies Dataset. (https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset)
2. Вибрати колонки: title та overview (короткий опис).
3. Для нового фільму знайти 3 фільми з найбільш схожим змістом.
4. Вивести результати у вигляді таблиці: назва та опис.

In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os
import warnings

# --- 0. Налаштування ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 1. Завантажимо дані (Крок 1) ---
file_path = os.path.join('DataSet', 'movies_metadata.csv')
df_full = None

try:
    # low_memory=False - щоб уникнути DtypeWarning, 
    # usecols - щоб завантажити тільки потрібні колонки (Крок 2)
    df_full = pd.read_csv(
        file_path, 
        usecols=['title', 'overview'], 
        low_memory=False
    )
    print(f"Файл '{file_path}' успішно завантажено. Всього: {len(df_full)} фільмів.")

except FileNotFoundError:
    print(f"ПОМИЛКА: Файл не знайдено за шляхом {file_path}")
    print("Переконайтеся, що папка 'DataSet' знаходиться поруч з вашим файлом Jupyter.")
except Exception as e:
    print(f"Помилка читання файлу: {e}")

if df_full is not None:
    
    # 2. Проведемо очищення та Семплінг 
    df_full.dropna(subset=['title', 'overview'], inplace=True)
    print(f"Після очищення NaN в 'overview': {len(df_full)} фільмів.")

    # ВАЖЛИВО: Датасет великий (45k+). 
    # Створення TF-IDF матриці для всіх може бути повільним.
    # Візьмемо репрезентативний семпл (наприклад, 20,000)
    df_sample = df_full.sample(n=20000, random_state=42)
    print(f"Взято семпл: {len(df_sample)} фільмів.")

    # --- 3. Проведемо симуляцію "Нового фільму" (Крок 3) ---
    # (Для прикладу візьмемо фільм про космос)
    new_movie = {
        'title': 'Space Odyssey 2025',
        'overview': 'A lone astronaut travels through space to find a new home for humanity. A mission to save mankind.'
    }
    
    # Додаємо його до нашого семплу
    new_movie_df = pd.DataFrame([new_movie])
    df_combined = pd.concat([df_sample, new_movie_df], ignore_index=True)

    # 4. Проведемо векторизацію (TF-IDF)
    # (Використовуємо stop_words='english', оскільки описи англійською)
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(df_combined['overview'])
    print(f"TF-IDF матрицю створено. Розмір: {tfidf_matrix.shape}")

    # 5. Проведемо розрахунок схожості (Cosine Similarity)
    similarity = cosine_similarity(tfidf_matrix)
    print("Матрицю схожості розраховано.")

    # 6. Створимо функцію 
    def recommend_movie_content_based(title, data_frame, similarity_matrix, n=3):
        if title not in data_frame['title'].values:
            return "Такого фільму немає в базі."
        
        idx = data_frame[data_frame['title'] == title].index[0]
        sim_scores = list(enumerate(similarity_matrix[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:n+1] # Пропускаємо [0] (сам фільм)
        movie_indices = [i[0] for i in sim_scores]
        
        # Використовуємо 'title' та 'overview' (Крок 2)
        return data_frame.iloc[movie_indices][['title', 'overview']]

    # --- 7. Отримаємо результат (Крок 4) ---
    print(f"\n--- 3 схожі фільми для (Новий фільм): '{new_movie['title']}' ---")
    recommendations = recommend_movie_content_based(new_movie['title'], df_combined, similarity, n=3)
    
    # Виведимо як таблицю (використовуючи .to_string(), щоб уникнути помилки 'tabulate')
    print(recommendations.to_string())

else:
    print("Завантаження даних не вдалося. Виконання зупинено.")

Файл 'DataSet\movies_metadata.csv' успішно завантажено. Всього: 45466 фільмів.
Після очищення NaN в 'overview': 44506 фільмів.
Взято семпл: 20000 фільмів.
TF-IDF матрицю створено. Розмір: (20001, 50945)
Матрицю схожості розраховано.

--- 3 схожі фільми для (Новий фільм): 'Space Odyssey 2025' ---
                           title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       overview
3908   Iron Sky: The Coming Race  Twenty years after the events of Iron Sky, the former Nazi Moonbase has become the last refuge of mankind. Earth was devastat

### Завдання 2. Рекомендації для користувача на основі історії оцінок
- Ситуація: Стримінговий сервіс хоче порадити користувачу нові фільми на основі його оцінок.
- Кроки:
1. Завантажити датасет MovieLens 100K (https://www.kaggle.com/datasets/prajitdatta/movielens-100k-dataset).
2. Створити таблицю користувач × фільм з рейтингами (0 = не дивився).
3. Для конкретного користувача визначити топ-3 фільми, які він ще не оцінював, і які, ймовірно, йому сподобаються.
4. Пояснити логіку, чому обрано саме ці фільми.

In [4]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import os

# --- 1. Визначимо правильно шлях до файлів в датасетів ---
base_path = os.path.join('DataSet', 'MovieLens 100K')
ratings_path = os.path.join(base_path, 'u.data')
movies_path = os.path.join(base_path, 'u.item')

# --- 2. Завантажимо дані ---
try:
    ratings_cols = ['user_id', 'item_id', 'rating', 'timestamp']
    ratings_df = pd.read_csv(ratings_path, sep='\t', names=ratings_cols)
    
    movies_cols = ['item_id', 'title']
    movies_df = pd.read_csv(movies_path, sep='|', names=movies_cols, encoding='latin-1', usecols=[0, 1])
    
    print(f"Файли успішно завантажено з: \n{ratings_path}\n{movies_path}")
    
except FileNotFoundError:
    print(f"ПОМИЛКА: Файли не знайдено за вказаними шляхами.")
    print(f"Перевірте, що папка 'DataSet' знаходиться поруч з вашим файлом Jupyter.")
    exit()

# --- 3. Створимо матриці User x Item ---
ratings_matrix = ratings_df.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)
print(f"\nМатрицю User x Item створено. Розмір: {ratings_matrix.shape}")

# --- 4. Розрахуємо схожості користувачів (User-Based) ---
user_similarity = cosine_similarity(ratings_matrix)
user_sim_df = pd.DataFrame(user_similarity, index=ratings_matrix.index, columns=ratings_matrix.index)
print("Матрицю схожості користувачів розраховано.")

# --- 5. Створимо функцію рекомендацій  ---
def recommend_for_user_id(user_id, ratings_matrix, user_sim_df, n=3):
    if user_id not in ratings_matrix.index:
        return "Такого користувача немає."

    similar_users = user_sim_df[user_id].sort_values(ascending=False)
    similar_users = similar_users[1:] 

    weighted_scores = pd.Series(0, index=ratings_matrix.columns, dtype=float)
    similarity_sum = pd.Series(0, index=ratings_matrix.columns, dtype=float)

    for other_user_id, sim in similar_users.items():
        if sim <= 0: continue 
        for item_id in ratings_matrix.columns:
            if ratings_matrix.loc[other_user_id, item_id] > 0:
                weighted_scores[item_id] += sim * ratings_matrix.loc[other_user_id, item_id]
                similarity_sum[item_id] += sim
    
    predicted_ratings = weighted_scores / similarity_sum
    predicted_ratings = predicted_ratings.fillna(0)

    unseen_mask = ratings_matrix.loc[user_id] == 0
    recommendations = predicted_ratings[unseen_mask].sort_values(ascending=False)
    
    return recommendations.head(n)

# --- 6. Отримаємо Топ-3 рекомендацій для User 1 (для прикладу) ---
USER_TO_RECOMMEND = 1 
top_3_recs = recommend_for_user_id(USER_TO_RECOMMEND, ratings_matrix, user_sim_df, n=3)

rec_df = top_3_recs.reset_index()
rec_df.columns = ['item_id', 'predicted_rating']

rec_with_titles = rec_df.merge(movies_df, on='item_id')

print(f"\n--- Топ-3 рекомендації для User {USER_TO_RECOMMEND} ---")

print(rec_with_titles[['title', 'item_id', 'predicted_rating']])

# --- 7. Пояснення логіки ---
print(f"\n--- Пояснення логіки для User {USER_TO_RECOMMEND} ---")
print(f"Ці рекомендації базуються на оцінках користувачів, схожих на User {USER_TO_RECOMMEND}.")

rec_item_ids = rec_with_titles['item_id'].tolist()

for item_id in rec_item_ids:
    title = movies_df[movies_df['item_id'] == item_id]['title'].values[0]
    print(f"\nФільм: '{title}' (ID: {item_id})")
    print(f"  > Ви його ще не бачили (Ваш рейтинг: 0).")
    
    raters = ratings_matrix[ratings_matrix[item_id] > 0].index
    similar_raters = user_sim_df.loc[USER_TO_RECOMMEND, raters].sort_values(ascending=False)
    top_similar_raters = similar_raters[similar_raters > 0.3].head(3) 
    
    if top_similar_raters.empty:
        print(f"  > Рекомендовано через сукупні оцінки багатьох користувачів.")
    else:
        print(f"  > Він дуже сподобався користувачам, схожим на Вас:")
        for user_id, sim in top_similar_raters.items():
            rating = ratings_matrix.loc[user_id, item_id]
            print(f"    - User {user_id} (схожість: {sim:.2f}) оцінив його на {rating:.0f}/5.")

Файли успішно завантажено з: 
DataSet\MovieLens 100K\u.data
DataSet\MovieLens 100K\u.item

Матрицю User x Item створено. Розмір: (943, 1682)
Матрицю схожості користувачів розраховано.

--- Топ-3 рекомендації для User 1 ---
                                               title  item_id  \
0                          Santa with Muscles (1996)     1500   
1               Saint of Fort Washington, The (1993)     1467   
2  Entertaining Angels: The Dorothy Day Story (1996)     1653   

   predicted_rating  
0               5.0  
1               5.0  
2               5.0  

--- Пояснення логіки для User 1 ---
Ці рекомендації базуються на оцінках користувачів, схожих на User 1.

Фільм: 'Santa with Muscles (1996)' (ID: 1500)
  > Ви його ще не бачили (Ваш рейтинг: 0).
  > Він дуже сподобався користувачам, схожим на Вас:
    - User 279 (схожість: 0.42) оцінив його на 5/5.

Фільм: 'Saint of Fort Washington, The (1993)' (ID: 1467)
  > Ви його ще не бачили (Ваш рейтинг: 0).
  > Він дуже сподобався ко

### Завдання 3. Пошук схожих продуктів у магазині
- Ситуація: Онлайн-магазин хоче запропонувати клієнтам товари, схожі на ті, що вони вже купували.
- Кроки:
1. Використати датасет Online Retail II Dataset (https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci).
2. Побудувати матрицю продукти × користувачі або покупки.
3. Для обраного товару знайти 5 найбільш схожих товарів.
4. Показати результати у вигляді таблиці: назва товару, категорія.

In [9]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import os
import warnings

# --- 0. Налаштування ---
# (Приховуємо попередження, вони не є помилками)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 1. Завантажимо дані ---
file_path = os.path.join('DataSet', 'Online Retail II Dataset.csv')

correct_cols = ['Customer ID', 'StockCode', 'Description', 'Quantity']

df = None # Ініціалізуємо df

try:
    df = pd.read_csv(
        file_path, 
        encoding='latin1', 
        usecols=correct_cols
    )
    print(f"Файл '{file_path}' успішно завантажено.")
    
except FileNotFoundError:
    print(f"ПОМИЛКА: Файл не знайдено за шляхом {file_path}")
except ValueError as e:
    print(f"ПОМИЛКА (ValueError): {e}. Схоже, назви колонок у файлі знову не збігаються.")
except Exception as e:
    print(f"Загальна помилка при читанні файлу: {e}")

# --- 2. Підготуємо дані та створимо матриці ---
if df is not None:
    
    # 2.1. Очищемо дані
    df.dropna(subset=['Customer ID'], inplace=True)
    df = df[df['Quantity'] > 0]
    
    df.dropna(subset=['StockCode'], inplace=True)
    
    non_product_codes = ['POST', 'DOT', 'M', 'MANUAL', 'C2', 'BANK CHARGES', 'PADS', 'D', 'CRUK']
    df = df[~df['StockCode'].isin(non_product_codes)]
    
    if 'Customer ID' in df.columns and not df.empty:
        df['Customer ID'] = df['Customer ID'].astype(int)
    else:
        print("Помилка: Дані порожні після очищення 'Customer ID'.")
        df = None # Зупиняємо виконання

# (Перевірка, що дані все ще валідні)
if df is not None:
    
    # 2.2. Оптимізуємо (Зменшення розміру матриці)
    top_users = df['Customer ID'].value_counts().head(1000).index
    df_filtered = df[df['Customer ID'].isin(top_users)]

    top_products = df_filtered['StockCode'].value_counts().head(1000).index
    df_filtered = df_filtered[df_filtered['StockCode'].isin(top_products)]

    if df_filtered.empty:
        print("Помилка: Фільтрація не дала результатів (DataFrame порожній).")
    else:
        print(f"Дані відфільтровано: {len(df_filtered)} записів, {df_filtered['StockCode'].nunique()} товарів, {df_filtered['Customer ID'].nunique()} користувачів.")
        
        # 2.3. Створимо матриці "Продукти × Користувачі" (Крок 2)
        product_user_matrix = df_filtered.pivot_table(
            index='StockCode', 
            columns='Customer ID', 
            values='Quantity', 
            aggfunc='sum'
        ).fillna(0)

        print(f"Матрицю 'Продукт x Користувач' створено. Розмір: {product_user_matrix.shape}")

        # --- 3. Розрахуємо схожі товари ---
        item_similarity = cosine_similarity(product_user_matrix)

        item_sim_df = pd.DataFrame(
            item_similarity, 
            index=product_user_matrix.index, 
            columns=product_user_matrix.index
        )

        print("Схожість між товарами розраховано.")

        # --- 4. Створимо мапи для назв товарів ---
        description_map = df_filtered.drop_duplicates(subset=['StockCode']).set_index('StockCode')['Description']

        # --- 5. Створимо функцію рекомендацій (Крок 3) ---
        def recommend_similar_products(stock_code, sim_matrix, mapping, n=5):
            if stock_code not in sim_matrix.index:
                return f"Товар {stock_code} не знайдено в матриці (можливо, він був відфільтрований)."
            
            sim_scores = sim_matrix[stock_code].sort_values(ascending=False)
            top_similar = sim_scores.iloc[1:n+1].reset_index()
            top_similar.columns = ['StockCode', 'Similarity']
            
            # (ВИМОГА КРОКУ 4: назва товару)
            top_similar['Description'] = top_similar['StockCode'].map(mapping)
            
            # (Примітка: Категорії немає в CSV, тому повертаємо 'Description')
            return top_similar[['StockCode', 'Description', 'Similarity']]

        # --- 6. Виконануємо та виводимо результати (Крок 4) ---
        TEST_STOCK_CODE = '85123A' # "WHITE HANGING HEART T-LIGHT HOLDER"

        if TEST_STOCK_CODE not in item_sim_df.index:
            if not item_sim_df.empty:
                TEST_STOCK_CODE = item_sim_df.index[0]
                print(f"INFO: Тестовий код '85123A' не знайдено, використано перший доступний: {TEST_STOCK_CODE}")
            else:
                TEST_STOCK_CODE = None
        
        if TEST_STOCK_CODE:
            product_name = description_map.get(TEST_STOCK_CODE, 'N/A')
            print(f"\n--- 5 Схожих товарів для: {TEST_STOCK_CODE} ({product_name}) ---")
            
            recommendations = recommend_similar_products(TEST_STOCK_CODE, item_sim_df, description_map, n=5)
            
            # (Використаємо .to_string(), щоб уникнути помилки 'tabulate')
            print(recommendations.to_string(index=False))
        else:
            print("Не вдалося виконати рекомендації: матриця порожня.")
else:
    print("Завантаження даних не вдалося. Подальше виконання зупинено.")

Файл 'DataSet\Online Retail II Dataset.csv' успішно завантажено.
Дані відфільтровано: 380685 записів, 1000 товарів, 1000 користувачів.
Матрицю 'Продукт x Користувач' створено. Розмір: (1000, 1000)
Схожість між товарами розраховано.

--- 5 Схожих товарів для: 85123A (WHITE HANGING HEART T-LIGHT HOLDER) ---
StockCode                       Description  Similarity
    21733  RED HANGING HEART T-LIGHT HOLDER    0.623158
    71477 COLOUR GLASS. STAR T-LIGHT HOLDER    0.622485
    21485   RED SPOT HEART HOT WATER BOTTLE    0.606608
    22637             PIGGY BANK RETROSPOT     0.592707
    22804   CANDLEHOLDER PINK HANGING HEART    0.565184


### Завдання 4. Аналіз рекомендацій для одного користувача
- Ситуація: Користувач сервісу дивився кілька фільмів, потрібно зрозуміти, які нові фільми йому будуть цікаві, використовуючи різні дані.
- Кроки:
1. Завантажити датасети:
    - MovieLens 100K (https://www.kaggle.com/datasets/prajitdatta/movielens-100k-dataset)
    - The Movies Dataset (https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset)
2. Для одного користувача скласти два списки рекомендованих фільмів:
    - на основі описів фільмів, які він оцінив високо,
    - на основі схожості користувачів із сервісу.
3. Порівняти списки, зробити висновки про відмінності.

In [13]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import os
import re
import warnings

# --- 0. Налаштування ---
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_colwidth', 100)

# --- 1. Визначимо шляхи та завантаження даних ---

ml_path = os.path.join('DataSet', 'MovieLens 100K')
meta_path = os.path.join('DataSet', 'movies_metadata.csv')

# 1.1. Завантаження MovieLens 100K
ratings_df = None
ml_movies_df = None
meta_df = None

try:
    ratings_cols = ['user_id', 'item_id', 'rating', 'timestamp']
    ratings_df = pd.read_csv(os.path.join(ml_path, 'u.data'), sep='\t', names=ratings_cols)
    
    movies_cols = ['item_id', 'title']
    ml_movies_df = pd.read_csv(os.path.join(ml_path, 'u.item'), sep='|', names=movies_cols, encoding='latin1', usecols=[0, 1])
    
    print(f"MovieLens 100K (u.data, u.item) завантажено.")
except FileNotFoundError:
    print(f"ПОМИЛКА: Не вдалося знайти файли MovieLens 100K у папці '{ml_path}'")
    try:
        print("Спроба альтернативного шляху (припускаючи, що Notebook всередині DataSet)...")
        ml_path = 'MovieLens 100K' # Без DataSet/
        ratings_df = pd.read_csv(os.path.join(ml_path, 'u.data'), sep='\t', names=ratings_cols)
        ml_movies_df = pd.read_csv(os.path.join(ml_path, 'u.item'), sep='|', names=movies_cols, encoding='latin1', usecols=[0, 1])
        print(f"MovieLens 100K (u.data, u.item) завантажено з '{ml_path}'.")
    except FileNotFoundError:
        print(f"ПОМИЛКА: Не вдалося знайти файли MovieLens і там.")
        ratings_df = None # Залишаємо None, щоб зупинити виконання

# 1.2. Завантажимо The Movies Dataset (для описів)
if ratings_df is not None: # Продовжуємо, лише якщо ML завантажено
    try:
        meta_df = pd.read_csv(meta_path, usecols=['title', 'overview'], low_memory=False)
        print(f"The Movies Dataset (movies_metadata.csv) завантажено з '{meta_path}'.")
    except FileNotFoundError:
        print(f"ПОМИЛКА: Не вдалося знайти '{meta_path}'")
        # Спробуємо альтернативний шлях
        try:
            meta_path = 'movies_metadata.csv' # Без DataSet/
            meta_df = pd.read_csv(meta_path, usecols=['title', 'overview'], low_memory=False)
            print(f"The Movies Dataset (movies_metadata.csv) завантажено з '{meta_path}'.")
        except FileNotFoundError:
             print(f"ПОМИЛКА: Не вдалося знайти '{meta_path}' і там.")
             meta_df = None # Зупиняємо

# --- (Головна перевірка) ---
# Виконуємо код, лише якщо ОБИДВА набори даних успішно завантажені
if ratings_df is not None and ml_movies_df is not None and meta_df is not None:

    # --- 2. Проведемо злиття даних (MovieLens <-> Metadata) ---
    meta_df.dropna(subset=['title', 'overview'], inplace=True)
    meta_df.drop_duplicates(subset=['title'], inplace=True, keep='first')

    def clean_title(title):
        return re.sub(r'\s*\(\d{4}\)$', '', title).strip()

    ml_movies_df['title_clean'] = ml_movies_df['title'].apply(clean_title)

    movies_merged_df = ml_movies_df.merge(meta_df, left_on='title_clean', right_on='title', how='inner')
    overview_map = movies_merged_df.set_index('item_id')['overview']
    title_map = ml_movies_df.set_index('item_id')['title']

    print(f"Знайдено {len(movies_merged_df)} збігів між MovieLens та Metadata.")

    # --- 3. Обиремо користувача ---
    USER_ID = 1
    print(f"\n--- Аналіз для User {USER_ID} ---")

    # ==============================================================================
    # МЕТОД 1: На основі схожості користувачів (Collaborative Filtering)
    # ==============================================================================
    print("\nМетод 1: Рекомендації на основі СХОЖИХ КОРИСТУВАЧІВ (CF)...")

    ratings_matrix = ratings_df.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)
    user_similarity = cosine_similarity(ratings_matrix)
    user_sim_df = pd.DataFrame(user_similarity, index=ratings_matrix.index, columns=ratings_matrix.index)

    def recommend_for_user_id(user_id, ratings_matrix, user_sim_df, n=3):
        similar_users = user_sim_df[user_id].sort_values(ascending=False)[1:]
        weighted_scores = pd.Series(0, index=ratings_matrix.columns, dtype=float)
        similarity_sum = pd.Series(0, index=ratings_matrix.columns, dtype=float)

        for other_user_id, sim in similar_users.items():
            if sim <= 0.3: continue 
            for item_id in ratings_matrix.columns:
                if ratings_matrix.loc[other_user_id, item_id] > 0:
                    weighted_scores[item_id] += sim * ratings_matrix.loc[other_user_id, item_id]
                    similarity_sum[item_id] += sim
        
        predicted_ratings = weighted_scores / similarity_sum
        predicted_ratings = predicted_ratings.fillna(0)
        unseen_mask = ratings_matrix.loc[user_id] == 0
        recommendations = predicted_ratings[unseen_mask].sort_values(ascending=False)
        return recommendations.head(n)

    collab_recs = recommend_for_user_id(USER_ID, ratings_matrix, user_sim_df, n=3)
    collab_recs_df = collab_recs.reset_index()
    collab_recs_df.columns = ['item_id', 'predicted_rating']
    collab_recs_df['title'] = collab_recs_df['item_id'].map(title_map)
    collab_recs_df = collab_recs_df[['title', 'predicted_rating']]

    # ==============================================================================
    # МЕТОД 2: На основі описів фільмів (Content-Based)
    # ==============================================================================
    print("\nМетод 2: Рекомендації на основі СХОЖИХ ОПИСІВ (Content)...")

    user_ratings = ratings_df[ratings_df['user_id'] == USER_ID]
    user_favorites = user_ratings[user_ratings['rating'] == 5]
    user_seen = user_ratings['item_id'].unique() 

    fav_overviews = movies_merged_df[movies_merged_df['item_id'].isin(user_favorites['item_id'])]['overview']

    if fav_overviews.empty:
        print(f"ПОМИЛКА: Не знайдено описів для улюблених фільмів User {USER_ID}.")
        content_recs_df = pd.DataFrame(columns=['title', 'similarity'])
    else:
        user_profile_text = " ".join(fav_overviews)
        
        content_data = meta_df[['title', 'overview']]
        profile_entry = pd.DataFrame([{'title': '_USER_PROFILE_', 'overview': user_profile_text}])
        content_data_combined = pd.concat([profile_entry, content_data], ignore_index=True)

        vectorizer = TfidfVectorizer(stop_words='english', max_features=5000) 
        tfidf_matrix = vectorizer.fit_transform(content_data_combined['overview'])
        
        print("TF-IDF матрицю (Content-Based) створено.")

        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])
        
        sim_scores = list(enumerate(similarity[0]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        content_recs_list = []
        
        seen_titles = ml_movies_df[ml_movies_df['item_id'].isin(user_seen)]['title_clean'].unique()
        
        for idx, score in sim_scores:
            if len(content_recs_list) >= 3:
                break 
            
            title = content_data_combined.iloc[idx + 1]['title'] 
            
            if title not in seen_titles:
                content_recs_list.append({'title': title, 'similarity': score})

        content_recs_df = pd.DataFrame(content_recs_list)

    # --- 5. Проведемо порівняння результатів ---
    print("\n" + "="*50)
    print(f"РЕЗУЛЬТАТИ РЕКОМЕНДАЦІЙ ДЛЯ USER {USER_ID}")
    print("="*50)

    print("\nМетод 1: На основі СХОЖИХ КОРИСТУВАЧІВ (Колаборативна фільтрація)")
    print("(Фільми, які сподобались вашим 'близнюкам' по оцінках)")
    print(collab_recs_df.to_string(index=False))

    print("\nМетод 2: На основі СХОЖОГО ЗМІСТУ (Content-Based)")
    print("(Фільми з описом, схожим на ваші улюблені)")
    print(content_recs_df.to_string(index=False))

    print("\n--- Висновки ---")
    if collab_recs_df.empty and content_recs_df.empty:
        print("Не вдалося згенерувати рекомендації жодним методом.")
    elif collab_recs_df.equals(content_recs_df):
        print("Висновки: Обидва методи дивним чином дали однакові результати.")
    else:
        print("Висновки: Списки рекомендацій відрізняються.")
        print("1. Колаборативна фільтрація (Метод 1) знайшла фільми, які популярні серед користувачів зі схожими смаками.")
        print("2. Content-Based (Метод 2) знайшов фільми, які тематично (за описом) схожі на ті, що User 1 вже оцінив високо.")

else:
    print("\nЗавантаження одного або обох файлів не вдалося. Виконання зупинено.")

MovieLens 100K (u.data, u.item) завантажено.
The Movies Dataset (movies_metadata.csv) завантажено з 'DataSet\movies_metadata.csv'.
Знайдено 1123 збігів між MovieLens та Metadata.

--- Аналіз для User 1 ---

Метод 1: Рекомендації на основі СХОЖИХ КОРИСТУВАЧІВ (CF)...

Метод 2: Рекомендації на основі СХОЖИХ ОПИСІВ (Content)...
TF-IDF матрицю (Content-Based) створено.

РЕЗУЛЬТАТИ РЕКОМЕНДАЦІЙ ДЛЯ USER 1

Метод 1: На основі СХОЖИХ КОРИСТУВАЧІВ (Колаборативна фільтрація)
(Фільми, які сподобались вашим 'близнюкам' по оцінках)
                                     title  predicted_rating
Marlene Dietrich: Shadow and Light (1996)                5.0
                      Aiqing wansui (1994)               5.0
                         Angel Baby (1995)               5.0

Метод 2: На основі СХОЖОГО ЗМІСТУ (Content-Based)
(Фільми з описом, схожим на ваші улюблені)
                        title  similarity
The Star Wars Holiday Special    0.213137
    The Scent of Green Papaya    0.190351
          

### Завдання 5. Інтеграція нового користувача або нового товару
- Ситуація: До платформи приєднався новий користувач або додався новий товар. Потрібно знайти йому рекомендації.
- Кроки:
1. Використати будь-який з датасетів попередніх завдань.
2. Додати нового користувача з кількома оцінками або новий товар із описом.
3. Для нового користувача запропонувати топ-3 фільми/товари, які він може оцінити високо.
4. Пояснити, які фактори вплинули на вибір рекомендацій.

In [16]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import os
import warnings

# --- 0. Налаштування ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 1. Завантажимо дані ---
base_path = os.path.join('DataSet', 'MovieLens 100K')
ratings_path = os.path.join(base_path, 'u.data')
movies_path = os.path.join(base_path, 'u.item')

try:
    ratings_cols = ['user_id', 'item_id', 'rating', 'timestamp']
    ratings_df = pd.read_csv(ratings_path, sep='\t', names=ratings_cols)
    
    movies_cols = ['item_id', 'title']
    movies_df = pd.read_csv(movies_path, sep='|', names=movies_cols, encoding='latin1', usecols=[0, 1])
    
    print(f"Файли MovieLens 100K успішно завантажено.")
    
except FileNotFoundError:
    print(f"ПОМИЛКА: Файли не знайдено у '{base_path}'")
    exit()

# --- 2. Створимо матриці User x Item ---
ratings_matrix = ratings_df.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)
print(f"Матрицю User x Item створено. Розмір: {ratings_matrix.shape}")

# --- 3. (Крок 1) Додамо Нового Користувача ---
NEW_USER_ID = 944

new_user_ratings = {
    50: 5,  # Star Wars (1977)
    172: 5, # Empire Strikes Back, The (1980)
    181: 5  # Return of the Jedi (1983)
}

new_user_series = pd.Series(new_user_ratings, name=NEW_USER_ID)
ratings_matrix.loc[NEW_USER_ID] = new_user_series.reindex(ratings_matrix.columns, fill_value=0)

print(f"Нового користувача (ID {NEW_USER_ID}) додано до матриці.")
print(f"Новий розмір матриці: {ratings_matrix.shape}")

# --- 4. Розрахуємо схожості (Включаючи нового користувача) ---
user_similarity = cosine_similarity(ratings_matrix)
user_sim_df = pd.DataFrame(user_similarity, index=ratings_matrix.index, columns=ratings_matrix.index)
print("Матрицю схожості оновлено.")

# --- 5. Створимо функцію рекомендацій ---
def recommend_for_user_id(user_id, ratings_matrix, user_sim_df, n=3):
    if user_id not in ratings_matrix.index:
        return "Такого користувача немає."

    similar_users = user_sim_df[user_id].sort_values(ascending=False)
    similar_users = similar_users[1:] 

    weighted_scores = pd.Series(0, index=ratings_matrix.columns, dtype=float)
    similarity_sum = pd.Series(0, index=ratings_matrix.columns, dtype=float)

    for other_user_id, sim in similar_users.items():
        if sim <= 0.3: continue 
        for item_id in ratings_matrix.columns:
            if ratings_matrix.loc[other_user_id, item_id] > 0:
                weighted_scores[item_id] += sim * ratings_matrix.loc[other_user_id, item_id]
                similarity_sum[item_id] += sim
    
    predicted_ratings = weighted_scores / similarity_sum
    predicted_ratings = predicted_ratings.fillna(0)

    unseen_mask = ratings_matrix.loc[user_id] == 0
    recommendations = predicted_ratings[unseen_mask].sort_values(ascending=False)
    
    return recommendations.head(n)

# --- 6. (Крок 2) Отримаємо Топ-3 рекомендацій ---
top_3_recs = recommend_for_user_id(NEW_USER_ID, ratings_matrix, user_sim_df, n=3)

rec_df = top_3_recs.reset_index()
rec_df.columns = ['item_id', 'predicted_rating']

rec_with_titles = rec_df.merge(movies_df, on='item_id')

print(f"\n--- (Крок 3) Топ-3 рекомендації для Нового Користувача (ID {NEW_USER_ID}) ---")
print(rec_with_titles[['title', 'item_id', 'predicted_rating']].to_string(index=False))
# -------------------------


# --- 7. (Крок 4) Пояснемо фактори ---
print(f"\n--- Пояснення логіки (Фактори вибору) ---")
print(f"Ці рекомендації базуються на оцінках користувачів, смаки яких СХОЖІ на смаки User {NEW_USER_ID} (який любить Sci-Fi).")

top_similar_users = user_sim_df[NEW_USER_ID].sort_values(ascending=False).index[1:4].tolist()
print(f"Найбільш схожі користувачі (близнюки): {top_similar_users}")

rec_item_ids = rec_with_titles['item_id'].tolist()

for item_id in rec_item_ids:
    title = movies_df[movies_df['item_id'] == item_id]['title'].values[0]
    print(f"\nФільм: '{title}' (ID: {item_id})")
    print(f"  > Ви його ще не бачили (Ваш рейтинг: 0).")
    
    print(f"  > Він сподобався вашим 'близнюкам':")
    for user_id in top_similar_users:
        rating = ratings_matrix.loc[user_id, item_id]
        if rating > 0:
            print(f"    - User {user_id} оцінив його на {rating:.0f}/5.")

Файли MovieLens 100K успішно завантажено.
Матрицю User x Item створено. Розмір: (943, 1682)
Нового користувача (ID 944) додано до матриці.
Новий розмір матриці: (944, 1682)
Матрицю схожості оновлено.

--- (Крок 3) Топ-3 рекомендації для Нового Користувача (ID 944) ---
                              title  item_id  predicted_rating
         Spitfire Grill, The (1996)      126               5.0
Hunchback of Notre Dame, The (1996)      596               5.0
                12 Angry Men (1957)      178               5.0

--- Пояснення логіки (Фактори вибору) ---
Ці рекомендації базуються на оцінках користувачів, смаки яких СХОЖІ на смаки User 944 (який любить Sci-Fi).
Найбільш схожі користувачі (близнюки): [51, 369, 182]

Фільм: 'Spitfire Grill, The (1996)' (ID: 126)
  > Ви його ще не бачили (Ваш рейтинг: 0).
  > Він сподобався вашим 'близнюкам':
    - User 182 оцінив його на 5/5.

Фільм: 'Hunchback of Notre Dame, The (1996)' (ID: 596)
  > Ви його ще не бачили (Ваш рейтинг: 0).
  > Він спод